In [1]:
import pandas as pd
import plotly.express as px

#RFM tulemuste laadimine.

In [2]:
rfm = pd.read_csv("rfm_results.csv")

rfm.shape

(2540, 10)

#Kliendisegmentide jaotus diagramm.

In [3]:
segment_count = rfm["segment"].value_counts().reset_index()

segment_count.columns = ["segment", "customers"]

fig = px.bar(
    segment_count,
    x="segment",
    y="customers",
    title="UrbanStyle kliendisegmentide jaotus",
    labels={
        "segment": "Kliendisegment",
        "customers": "Klientide arv"
    }
)

fig.show()

#Hajuvusdiagramm

Visualiseerime klientide ostukäitumist kasutades Recency, Frequency ja Monetary näitajaid.
Diagramm aitab tuvastada erinevaid kliendigruppe ja nende väärtust ettevõtte jaoks.

In [5]:
fig = px.scatter(
    rfm,
    x="recency_days",
    y="monetary",
    color="segment",
    size="frequency",
    title="UrbanStyle kliendisegmendid RFM analüüsi põhjal",
    labels={
        "recency_days": "Päevi viimasest ostust",
        "monetary": "Kogukulutus (EUR)",
        "frequency": "Ostude arv",
        "segment": "Kliendisegment"
    }
)

fig.show()

#Teen veidi paremini loetava diagrammi:

In [9]:
fig = px.box(
    rfm,
    x="segment",
    y="monetary",
    color="segment",
    title="UrbanStyle kliendisegmentide kulutuste võrdlus",
    labels={
        "segment": "Kliendisegment",
        "monetary": "Kogukulutus (€)"
    }
)

fig.show()

In [10]:
fig = px.scatter(
    rfm,
    x="recency_days",
    y="monetary",
    color="segment",
    size="frequency",
    opacity=0.6,
    title="UrbanStyle klientide RFM jaotus",
    labels={
        "recency_days": "Päevi viimasest ostust",
        "monetary": "Kogukulutus (€)",
        "segment": "Kliendisegment"
    }
)

fig.update_yaxes(type="log")

fig.show()

#See ülemine tundub kõige paremini loetavam, kuid andmed tunduvad olevat rasked, et teha sellest üldse loetav diagramm.

#VIP klientide analüüs

Leiame kõige väärtuslikumad VIP kliendid kogukulutuse põhjal.
See aitab tuvastada kliendid, kelle hoidmine on ettevõtte jaoks kõige olulisem.

In [11]:
vip = (
    rfm[rfm["segment"] == "VIP Champions"]
    .nlargest(10, "monetary")
)

vip

,customer_id,sale_date,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_Score,segment
1367,3618.0,2025-02-21,7,72,27920.86,5,5,5,15,VIP Champions
1136,3350.0,2025-02-09,19,76,26286.10,5,5,5,15,VIP Champions
751,2889.0,2026-05-17,-443,71,24172.58,5,5,5,15,VIP Champions
839,2997.0,2026-01-01,-307,77,23667.93,5,5,5,15,VIP Champions
1393,3648.0,2025-02-20,8,72,22942.42,5,5,5,15,VIP Champions
1870,4206.0,2025-02-28,0,69,21916.75,5,5,5,15,VIP Champions
1355,3605.0,2025-02-24,4,77,21626.10,5,5,5,15,VIP Champions
187,2221.0,2025-12-14,-289,65,21610.39,5,5,5,15,VIP Champions
128,2154.0,2025-02-18,10,66,20726.79,5,5,5,15,VIP Champions
1458,3722.0,2025-01-31,28,62,20317.81,5,5,5,15,VIP Champions


In [12]:
fig = px.bar(
    vip,
    x="customer_id",
    y="monetary",
    title="UrbanStyle Top 10 VIP klienti kogukulutuse järgi",
    labels={
        "customer_id": "Kliendi ID",
        "monetary": "Kogukulutus (€)"
    },
    color="monetary"
)

fig.show()

#Proovin teist varianti:

In [13]:
vip = (
    rfm[rfm["segment"] == "VIP Champions"]
    .nlargest(10, "monetary")
    .sort_values("monetary")
)

fig = px.bar(
    vip,
    x="monetary",
    y="customer_id",
    orientation="h",
    title="UrbanStyle Top 10 VIP klienti kogukulutuse järgi",
    labels={
        "customer_id": "Kliendi ID",
        "monetary": "Kogukulutus (€)"
    },
    color="monetary"
)

fig.show()

#Ikka pole rahul, proovin veel kahte diagrammi:

In [14]:
vip = (
    rfm[rfm["segment"] == "VIP Champions"]
    .nlargest(10, "monetary")
)

fig = px.treemap(
    vip,
    path=["customer_id"],
    values="monetary",
    title="UrbanStyle Top 10 VIP klienti kogukulutuse järgi",
    labels={
        "customer_id": "Kliendi ID",
        "monetary": "Kogukulutus (€)"
    }
)

fig.show()

#Scatter ainult vip klientidega:

In [15]:
vip = rfm[rfm["segment"] == "VIP Champions"]

fig = px.scatter(
    vip,
    x="frequency",
    y="monetary",
    size="monetary",
    hover_data=["customer_id"],
    title="VIP klientide ostukäitumine",
    labels={
        "frequency": "Ostude arv",
        "monetary": "Kogukulutus (€)"
    }
)

fig.show()

#mitu VIP klienti, kui suur käibeosakaal, mitu At-Risk klienti.

Need näitajad toetavad ärilisi soovitusi.

In [16]:
vip_count = (rfm["segment"] == "VIP Champions").sum()

at_risk_count = (rfm["segment"] == "At Risk").sum()

vip_revenue_share = (
    rfm[rfm["segment"] == "VIP Champions"]["monetary"].sum()
    / rfm["monetary"].sum()
    * 100
)

print("VIP kliente:", vip_count)
print("At Risk kliente:", at_risk_count)
print("VIP käibe osakaal:", round(vip_revenue_share, 2), "%")

VIP kliente: 455
At Risk kliente: 529
VIP käibe osakaal: 42.82 %


#Äritõlgendus Markole

RFM analüüsi põhjal kuulub VIP Champions segmenti 455 klienti, kes moodustavad ettevõtte jaoks kõige väärtuslikuma kliendirühma. Kuigi VIP kliendid moodustavad väiksema osa kogu kliendibaasist, annavad nad 42.82% kogu käibest.

At Risk segment sisaldab 529 klienti, kelle ostuaktiivsus on langenud ning kelle puhul on olemas risk kliendikaotuseks. Potential segment on suurim grupp, mis annab võimaluse kasvatada tulevikus rohkem lojaalseid kliente.

#Soovitused

1. VIP programm:
- pakkuda VIP klientidele personaalseid pakkumisi, eritingimusi ja varajast ligipääsu uutele toodetele.

2. Win-back kampaania:
- suunata At Risk klientidele personaalseid pakkumisi, soodustusi või meeldetuletusi, et taastada ostuaktiivsus.

3. Nurture programm:
- arendada Potential klientide gruppi läbi soovituste, kampaaniate ja personaalse kommunikatsiooni.

#Edasijõudnute tase - kaalutud RFM

#Kaalutud RFM skoor

Arvutan uue RFM skoori, kus Monetary väärtusel on kahekordne kaal.
See võimaldab rohkem esile tõsta suurema rahalise väärtusega kliente.

In [17]:
rfm["Weighted_RFM_Score"] = (
    rfm["R_score"].astype(int)
    + rfm["F_score"].astype(int)
    + (rfm["M_score"].astype(int) * 2)
)

rfm.head()

,customer_id,sale_date,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_Score,segment,Weighted_RFM_Score
0,2001.0,2024-11-29,91,2,203.92,4,1,1,6,At Risk,7
1,2004.0,2024-12-19,71,2,1198.56,4,1,4,9,Potential,13
2,2005.0,2024-10-03,148,4,959.60,3,4,3,10,Loyal,13
3,2006.0,2023-11-09,477,1,327.06,1,1,1,3,Lost,4
4,2007.0,2025-01-30,29,1,318.63,5,1,1,7,Potential,8


#Täiendatud kliendisegmendid

Loon detailsemad kliendisegmendid kaalutud RFM skoori põhjal.
Segmendid võimaldavad täpsemat turundustegevuste planeerimist.

In [18]:
def weighted_segment(score):
    if score >= 18:
        return "VIP Champions"
    elif score >= 14:
        return "Loyal Customers"
    elif score >= 10:
        return "Regular Customers"
    elif score >= 7:
        return "New Customers"
    elif score >= 5:
        return "At Risk"
    else:
        return "Lost"


rfm["weighted_segment"] = rfm["Weighted_RFM_Score"].apply(weighted_segment)

rfm.head()

,customer_id,sale_date,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_Score,segment,Weighted_RFM_Score,weighted_segment
0,2001.0,2024-11-29,91,2,203.92,4,1,1,6,At Risk,7,New Customers
1,2004.0,2024-12-19,71,2,1198.56,4,1,4,9,Potential,13,Regular Customers
2,2005.0,2024-10-03,148,4,959.60,3,4,3,10,Loyal,13,Regular Customers
3,2006.0,2023-11-09,477,1,327.06,1,1,1,3,Lost,4,Lost
4,2007.0,2025-01-30,29,1,318.63,5,1,1,7,Potential,8,New Customers


#Analüüsin, kuidas kliendid jagunevad uutesse segmentidesse ning hindan segmentide tasakaalu.

In [19]:
weighted_summary = (
    rfm["weighted_segment"]
    .value_counts()
    .reset_index()
)

weighted_summary.columns = [
    "Segment",
    "Klientide arv"
]

weighted_summary

,Segment,Klientide arv
0,Regular Customers,701
1,Loyal Customers,631
2,New Customers,496
3,VIP Champions,375
4,At Risk,219
5,Lost,118


#Salvestan lõplikud segmendid CSV faili, mida saab kasutada turundus- ja kliendihaldustegevustes.

In [20]:
rfm.to_csv(
    "rfm_segments.csv",
    index=False
)

print("Fail rfm_segments.csv salvestatud")

Fail rfm_segments.csv salvestatud


In [21]:
import os

os.path.exists("rfm_segments.csv")

True